# Pre-Training RQwen3 on FineWeb-Edu

## What is Pre-Training?

This notebook does **pre-training** — the very first stage of making a language model.
Right now our RQwen3 has completely random weights, so it outputs gibberish. Pre-training
is how we go from "random noise generator" to "something that actually understands language."

### The 3 stages of training a modern LLM

Real models like Qwen3, GPT-4, and LLaMA go through a pipeline:

| Stage | Name | What it does | Data | Result |
|-------|------|-------------|------|--------|
| **1. Pre-training** | Causal Language Modeling | Learns language from raw text | Trillions of tokens of web text | A model that can complete text and "understands" grammar, facts, reasoning |
| **2. SFT** | Supervised Fine-Tuning | Learns to follow instructions | Thousands of human-written Q&A pairs | A model that answers questions and follows directions |
| **3. RLHF / DPO** | Reinforcement Learning from Human Feedback | Learns human preferences | Human rankings of model outputs | A model that gives helpful, safe, high-quality answers |

**This notebook is Stage 1.** We feed raw text to the model and it learns to predict
the next token. That's it — but everything about language emerges from this single objective.

### What pre-training actually teaches the model

After enough pre-training, the model learns (in roughly this order):
1. **Token frequency** — which words are common vs rare
2. **Grammar** — English syntax, sentence structure
3. **Short-range patterns** — phrases, idioms, common word pairings
4. **Long-range coherence** — staying on topic across paragraphs
5. **World knowledge** — facts absorbed from the training text
6. **Reasoning patterns** — logical structures seen repeated in text

### What pre-training does NOT teach
- How to follow instructions ("Summarize this article")
- How to have a conversation (back-and-forth dialogue)
- How to be helpful, harmless, or honest
- When to say "I don't know"

Those come from Stage 2 (SFT) and Stage 3 (RLHF) — future notebooks!

### The training objective: Next-Token Prediction

The entire training objective is embarrassingly simple:

```
Input:  "The cat sat on the ___"
Target: "The cat sat on the mat"
                              ^ predict this
```

At every position in a sequence, the model tries to predict what comes next.
Cross-entropy loss measures how wrong it was. Gradients flow backward. Weights update.
Repeat this billions of times across trillions of tokens, and the model learns language.

This is called **Causal Language Modeling (CLM)** — "causal" because the model can only
look at tokens that came *before* the current position (it can't peek ahead).

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import time
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup

from src import (
    CoreConfig, RQwen3, get_device, format_param_count,
    TrainConfig, StreamingTokenDataset,
    save_checkpoint, load_checkpoint, save_snapshot, generate_sample,
)

device = get_device()
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

---
## Setup & Dataset

**Dataset: [FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu)**
- 1.3 trillion tokens of high-quality educational web text
- Each document is scored 0-5 for educational value (only high-scoring text is included)
- Created by HuggingFace — one of the best open pre-training datasets available

**Why this dataset?** Pre-training needs *lots* of diverse, high-quality text.
FineWeb-Edu is ideal because:
- It's massive (we'll never run out of data, even training for weeks)
- It's filtered for quality (less garbage = the model learns faster)
- It streams (no need to download 1.3T tokens to your laptop — data arrives on-demand)

**How to use this notebook:**
- **On Mac (MPS)**: Runs a quick test (~200 steps) to verify the pipeline works and see the loss start dropping
- **On desktop (CUDA)**: Run for 10,000+ steps for real training
- **After training**: Re-run notebooks 03 and 04 to see how the model changed

In [ ]:
train_cfg = TrainConfig()

# --- Auto-scale for Mac (MPS) vs Desktop (CUDA) ---
if device == 'mps':
    train_cfg.batch_size = 1
    train_cfg.seq_len = 512
    train_cfg.grad_accum_steps = 32
    train_cfg.max_steps = 200
    train_cfg.save_every = 100
    train_cfg.sample_every = 50
    train_cfg.warmup_steps = 20
    print('MPS detected -> using Mac-friendly settings (batch=1, seq=512)')
    print('For real training, move to your desktop (CUDA).')
elif device == 'cuda':
    print(f'CUDA detected -> using full training settings')
else:
    train_cfg.batch_size = 1
    train_cfg.seq_len = 256
    train_cfg.grad_accum_steps = 16
    train_cfg.max_steps = 50
    print('CPU detected -> minimal settings for testing only')

print()

os.makedirs(train_cfg.checkpoint_dir, exist_ok=True)
os.makedirs(train_cfg.snapshot_dir, exist_ok=True)

# Tokens per optimizer step
tokens_per_step = train_cfg.batch_size * train_cfg.grad_accum_steps * train_cfg.seq_len
total_tokens = tokens_per_step * train_cfg.max_steps
print(f'seq_len: {train_cfg.seq_len} | batch_size: {train_cfg.batch_size} | '
      f'grad_accum: {train_cfg.grad_accum_steps}')
print(f'Effective batch: {train_cfg.batch_size * train_cfg.grad_accum_steps} sequences = '
      f'{tokens_per_step:,} tokens/step')
print(f'Total tokens over {train_cfg.max_steps:,} steps: {format_param_count(total_tokens)}')

---
## Data Pipeline

### How raw text becomes training data

```
Raw text from FineWeb-Edu:
  "Parents are usually the first to recognize that their child has a problem..."

     |  Tokenizer (Qwen3 BPE, 151,936 vocab)
     v

Token IDs:
  [19281, 527, 6118, 279, 1156, 311, 15641, 430, 862, 1716, 702, 264, 3491, ...]

     |  Chunk into fixed-length sequences (seq_len tokens)
     v

Training example:
  input_ids: [19281, 527, 6118, 279, 1156, 311, ...]    <- tokens 0 to N-1
  labels:    [527, 6118, 279, 1156, 311, 15641, ...]    <- tokens 1 to N  (shifted by 1)
```

At every position, the model's job is to predict what comes next. The `labels` are just
the `input_ids` shifted by one position — this is how next-token prediction works.

Documents are separated by EOS (end-of-sequence) tokens so the model learns where
one document ends and the next begins.

In [ ]:
# Initialize
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B')
dataset = StreamingTokenDataset(train_cfg.dataset_name, tokenizer, train_cfg.seq_len)
dataloader = DataLoader(dataset, batch_size=train_cfg.batch_size)

# Quick test: grab one batch
test_batch = next(iter(dataloader))
print(f'Batch input_ids shape: {test_batch[0].shape}')
print(f'Batch labels shape:    {test_batch[1].shape}')
print(f'Sample tokens: {tokenizer.decode(test_batch[0][0][:30])}')
print('Data pipeline works.')

---
## Model & Optimizer

### Why AdamW?
AdamW is the standard optimizer for training transformers. It combines:
- **Adam**: Adapts the learning rate per-parameter (parameters that haven't been updated much get larger steps)
- **Weight decay**: Gently pushes weights toward zero to prevent overfitting (like L2 regularization, but done correctly for Adam)

### Why cosine learning rate schedule?
The learning rate controls how big each weight update is:
- **Too high**: training is unstable, loss spikes, weights explode
- **Too low**: training is painfully slow, model barely learns

The cosine schedule handles this with two phases:
1. **Warmup** (first ~500 steps): LR ramps up from ~0 to peak. This lets the model's randomly-initialized weights stabilize before we start making big updates
2. **Cosine decay** (rest of training): LR gradually decreases following a cosine curve. Early on we make big adjustments, later we fine-tune with smaller steps

### Weight decay: who gets it?
Not all parameters should be regularized:
- **Linear weight matrices** (attention projections, FFN layers): YES — these are the bulk of the model
- **Norm weights and biases**: NO — these are small scaling factors, decaying them hurts training

In [ ]:
# Create model
model_config = CoreConfig()
model = RQwen3(model_config).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: {total_params:,} params ({format_param_count(total_params)})')

# Optimizer: AdamW with weight decay on everything except norms and biases
decay_params = []
no_decay_params = []
for name, param in model.named_parameters():
    if param.dim() < 2 or 'norm' in name:  # biases, norm weights
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': decay_params, 'weight_decay': train_cfg.weight_decay},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=train_cfg.learning_rate)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=train_cfg.warmup_steps,
    num_training_steps=train_cfg.max_steps,
)

print(f'Optimizer: AdamW (lr={train_cfg.learning_rate}, wd={train_cfg.weight_decay})')
print(f'Schedule: cosine with {train_cfg.warmup_steps} warmup steps')
print(f'Decay params: {sum(p.numel() for p in decay_params):,}')
print(f'No-decay params: {sum(p.numel() for p in no_decay_params):,}')

In [ ]:
# Save untrained snapshot for later comparison
save_snapshot(model, 'untrained', train_cfg.snapshot_dir)
print('Untrained snapshot saved.')

---
## Training Loop

### What happens in one training step

```
   +-----------+     +----------+     +-----------+     +----------+
   | Text      | --> | Forward  | --> | Compute   | --> | Backward |
   | (tokens)  |     | Pass     |     | Loss      |     | Pass     |
   +-----------+     +----------+     +-----------+     +----------+
                                                              |
                                                              v
                     +----------+     +-----------+     +----------+
                     | Update   | <-- | Clip      | <-- | Accumulate
                     | Weights  |     | Gradients |     | Gradients|
                     +----------+     +-----------+     +----------+
```

1. **Forward pass**: Feed tokens through the model. At every position it outputs a probability distribution over all 151,936 possible next tokens
2. **Cross-entropy loss**: Compare the model's predictions to the actual next tokens. Higher loss = more wrong
3. **Backward pass**: Compute gradients — how should each weight change to reduce the loss?
4. **Gradient accumulation**: Repeat steps 1-3 for multiple micro-batches, adding up gradients. This simulates a larger batch without needing more GPU memory
5. **Gradient clipping**: Cap gradient magnitudes at 1.0 to prevent training instability
6. **Optimizer step**: AdamW uses the accumulated gradients to update all weights
7. **LR schedule step**: Adjust the learning rate according to the cosine schedule

### Gradient accumulation explained

We want a large effective batch (32 sequences) for stable gradients, but we can't
fit 32 sequences in GPU memory at once. Solution: process them in smaller micro-batches
and accumulate the gradients before updating.

```
Micro-batch 1: forward + backward (gradients stored, no weight update)
Micro-batch 2: forward + backward (gradients added to stored ones)
...
Micro-batch N: forward + backward (gradients added)
--> Now clip gradients, optimizer step, zero gradients, repeat
```

**To stop early**: Interrupt the kernel. The last checkpoint is saved automatically.

**To resume**: Set `RESUME_FROM` to the checkpoint path.

In [ ]:
# Set to a checkpoint path to resume, or None to start fresh
RESUME_FROM = None  # e.g. '../checkpoints/step_1000.pt'

# Resume or start fresh
if RESUME_FROM and os.path.exists(RESUME_FROM):
    start_step, loss_history = load_checkpoint(RESUME_FROM, model, optimizer, scheduler)
else:
    start_step = 0
    loss_history = []

SAMPLE_PROMPT = 'The most important thing about mathematics is'

# --- Training loop ---
model.train()
optimizer.zero_grad()

data_iter = iter(dataloader)
accum_loss = 0.0
data_step = 0
t_start = time.time()

print(f'Starting training from step {start_step}...')
print(f'Target: {train_cfg.max_steps:,} optimizer steps')
print(f'Effective batch: {train_cfg.batch_size} x {train_cfg.grad_accum_steps} = '
      f'{train_cfg.batch_size * train_cfg.grad_accum_steps} sequences')
print()

try:
    for opt_step in range(start_step, train_cfg.max_steps):
        # Accumulate gradients over multiple micro-batches
        for micro in range(train_cfg.grad_accum_steps):
            try:
                input_ids, labels = next(data_iter)
            except StopIteration:
                # Dataset exhausted (unlikely with 1.3T tokens), restart
                data_iter = iter(dataloader)
                input_ids, labels = next(data_iter)

            input_ids = input_ids.to(device)
            labels = labels.to(device)

            logits = model(input_ids)  # (batch, seq_len, vocab_size)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1),
            )
            loss = loss / train_cfg.grad_accum_steps
            loss.backward()
            accum_loss += loss.item()
            data_step += 1

        # Optimizer step
        torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.max_grad_norm)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        loss_history.append(accum_loss)
        accum_loss = 0.0

        # Logging
        if (opt_step + 1) % train_cfg.log_every == 0:
            elapsed = time.time() - t_start
            tokens_so_far = (opt_step + 1) * tokens_per_step
            tokens_per_sec = tokens_so_far / elapsed
            lr = scheduler.get_last_lr()[0]
            avg_loss = np.mean(loss_history[-train_cfg.log_every:])
            print(f'Step {opt_step+1:>6d}/{train_cfg.max_steps} | '
                  f'loss={avg_loss:.4f} | lr={lr:.2e} | '
                  f'{tokens_per_sec:,.0f} tok/s | '
                  f'{format_param_count(tokens_so_far)} tokens')

        # Sample generation
        if (opt_step + 1) % train_cfg.sample_every == 0:
            sample = generate_sample(model, tokenizer, SAMPLE_PROMPT)
            print(f'  Sample: "{sample[:120]}..."')
            model.train()

        # Checkpoint
        if (opt_step + 1) % train_cfg.save_every == 0:
            ckpt_path = os.path.join(train_cfg.checkpoint_dir, f'step_{opt_step+1}.pt')
            save_checkpoint(model, optimizer, scheduler, opt_step + 1, loss_history, ckpt_path)
            snap_path = save_snapshot(model, f'step_{opt_step+1}', train_cfg.snapshot_dir)
            print(f'  Snapshot saved: {snap_path}')

except KeyboardInterrupt:
    print(f'\nTraining interrupted at step {opt_step + 1}.')
    ckpt_path = os.path.join(train_cfg.checkpoint_dir, f'step_{opt_step+1}_interrupted.pt')
    save_checkpoint(model, optimizer, scheduler, opt_step + 1, loss_history, ckpt_path)
    print('Checkpoint saved. You can resume from this point.')

# Final save
else:
    ckpt_path = os.path.join(train_cfg.checkpoint_dir, f'step_{train_cfg.max_steps}_final.pt')
    save_checkpoint(model, optimizer, scheduler, train_cfg.max_steps, loss_history, ckpt_path)
    save_snapshot(model, 'trained', train_cfg.snapshot_dir)
    print(f'\nTraining complete. Final checkpoint: {ckpt_path}')

elapsed_total = time.time() - t_start
print(f'Total time: {elapsed_total/60:.1f} minutes')

---
## Training Progress

### How to read the loss curve

The loss tells you how surprised the model is by the correct answer. Think of it as a
"wrongness score":

- **~11.9** (ln(151,936)): Random guessing — the model has no idea, treats all 151,936 tokens as equally likely
- **~8-9**: The model learned which tokens are common (predicts "the", "is", "a" more often)
- **~5-6**: Grammar is forming — the model knows what types of words follow other words
- **~3-4**: Real language understanding — coherent phrases, stays on topic
- **~2-3**: Good language model — approaching the quality of small published models

**What to expect on Mac (200 steps):** You'll see the loss drop from ~11.9 to maybe ~9-10.
That's not much in terms of output quality, but it proves the pipeline works and the model
is learning. Real training on your desktop (10k+ steps) will show dramatic improvements.

In [ ]:
if loss_history:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    # Raw loss
    ax1.plot(loss_history, alpha=0.3, color='steelblue', linewidth=0.5)
    # Smoothed loss (rolling average)
    window = min(50, len(loss_history) // 5 or 1)
    if window > 1:
        smoothed = np.convolve(loss_history, np.ones(window)/window, mode='valid')
        ax1.plot(range(window-1, len(loss_history)), smoothed,
                 color='darkblue', linewidth=2, label=f'Smoothed ({window}-step avg)')
    ax1.set_xlabel('Optimizer Step')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
    ax1.legend()

    # Log scale
    ax2.plot(loss_history, alpha=0.3, color='steelblue', linewidth=0.5)
    if window > 1:
        ax2.plot(range(window-1, len(loss_history)), smoothed,
                 color='darkblue', linewidth=2)
    ax2.set_yscale('log')
    ax2.set_xlabel('Optimizer Step')
    ax2.set_ylabel('Loss (log scale)')
    ax2.set_title('Training Loss (Log Scale)', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(f'Initial loss: {loss_history[0]:.4f}')
    print(f'Final loss:   {loss_history[-1]:.4f}')
    print(f'Best loss:    {min(loss_history):.4f} (step {np.argmin(loss_history) + 1})')
    # Random baseline: -ln(1/vocab_size) = ln(vocab_size)
    random_loss = math.log(model_config.vocab_size)
    print(f'Random baseline: {random_loss:.4f} (ln(vocab_size={model_config.vocab_size:,}))')
else:
    print('No training history yet. Run the training loop first.')

In [ ]:
# Generate a few samples to see current model quality
prompts = [
    'The most important thing about mathematics is',
    'In computer science, algorithms are',
    'Once upon a time',
]

for prompt in prompts:
    output = generate_sample(model, tokenizer, prompt, max_new_tokens=80)
    print(f'Prompt: "{prompt}"')
    print(f'Output: {output}')
    print()

---
## After Training: Compare with Analysis Notebooks

If you saved snapshots during training, you can compare them using the tools
from notebook 03:

In [ ]:
# List saved snapshots
snapshot_files = sorted([f for f in os.listdir(train_cfg.snapshot_dir) if f.endswith('.pt')])
print('Saved snapshots:')
for f in snapshot_files:
    print(f'  {f}')

# Quick comparison: untrained vs latest
if 'untrained.pt' in snapshot_files and len(snapshot_files) > 1:
    latest = [f for f in snapshot_files if f != 'untrained.pt'][-1]
    snap_u = torch.load(os.path.join(train_cfg.snapshot_dir, 'untrained.pt'), weights_only=False)
    snap_t = torch.load(os.path.join(train_cfg.snapshot_dir, latest), weights_only=False)

    # How much did weights change?
    params = list(snap_u.keys())
    std_changes = [abs(snap_t[p]['std'] - snap_u[p]['std']) for p in params]
    mean_changes = [abs(snap_t[p]['mean'] - snap_u[p]['mean']) for p in params]

    print(f'\nComparing: untrained.pt vs {latest}')
    print(f'Avg |delta std|:  {np.mean(std_changes):.6f}')
    print(f'Avg |delta mean|: {np.mean(mean_changes):.6f}')
    print(f'Max |delta std|:  {np.max(std_changes):.6f}')
    print()
    print('For full visual comparison, open notebook 03 and run:')
    print(f'  compare_snapshots("untrained", "{latest.replace(".pt", "")}")')
else:
    print('\nTrain for at least one checkpoint to see comparisons.')

---
## What I Learned

### This notebook is Stage 1 of 3

We did **pre-training** (Causal Language Modeling) — the foundation that everything
else builds on. Here's where we are in the full pipeline:

```
[x] Stage 1: Pre-training (THIS NOTEBOOK)
    - Random weights -> language understanding
    - Objective: next-token prediction on raw text
    - Data: FineWeb-Edu (trillions of tokens)
    - Result: model that can complete text coherently

[ ] Stage 2: Supervised Fine-Tuning (SFT) -- future notebook 06
    - Pre-trained model -> instruction follower
    - Objective: learn from human-written Q&A examples
    - Data: instruction datasets (e.g., OpenAssistant, Alpaca)
    - Result: model that answers questions and follows directions

[ ] Stage 3: RLHF / DPO -- future notebook 07
    - Fine-tuned model -> aligned model
    - Objective: learn human preferences
    - Data: human rankings of model outputs
    - Result: model that gives helpful, high-quality answers
```

### Key Concepts

**1. Next-token prediction is the entire pre-training objective**
- The model sees tokens 1..N and tries to predict token 2..N+1
- Cross-entropy loss measures how far off the predictions are
- All of language understanding emerges from this single objective
- This is the same objective used to train GPT-4, Qwen3, LLaMA, etc.

**2. Loss tells you how much the model has learned**
- Random baseline: ln(vocab_size) ~ 11.9 (guessing uniformly)
- After pre-training: loss drops as the model learns patterns
- Lower loss = better next-token predictions = more language understanding

**3. Pre-training is computationally expensive**
- Real models train on trillions of tokens for weeks on GPU clusters
- Our 508M param model is small enough to train on consumer hardware
- Even a few thousand steps will show clear learning in the analysis notebooks

**4. Gradient accumulation simulates larger batches**
- Small GPU memory? Use batch_size=1 but accumulate 32 steps
- Equivalent to batch_size=32 for gradient quality
- Larger effective batches = smoother, more stable training

**5. Learning rate schedule matters**
- Warmup: start small to avoid destroying random init
- Cosine decay: gradually reduce LR as training converges
- Weight decay: regularization that keeps weights from growing too large

**6. Checkpointing is essential**
- Training can crash, get interrupted, or run out of memory
- Saving state (model + optimizer + scheduler) lets you resume exactly
- Snapshots let you track what changed at each stage

### Next Steps
- [ ] Run the Mac test (~200 steps) to verify the pipeline and see loss drop
- [ ] Move to desktop for real training (10,000+ steps on CUDA)
- [ ] Re-run notebook 03 to see how weight distributions and attention changed
- [ ] Re-run notebook 04 to compare pre-trained RQwen3 against real Qwen3-1.7B
- [ ] **Build notebook 06: SFT** — fine-tune the pre-trained model on instruction data
- [ ] **Build notebook 07: RLHF/DPO** — align the model with human preferences